# Kiyohara public-transport isochrone analysis

This notebook creates 30-minute and 60-minute public-transport accessibility polygons to the Kiyohara industrial-area LRT stop. It implements the requested travel-time workflow:

`area -> walk X min -> bus stop / rail node -> feeder bus Y min -> LRT stop -> LRT Z min -> destination`

Assumptions used by the workflow:

- `Z`: LRT travel time from each LRT stop to the destination, estimated from cumulative stop-to-stop route length in `lrt_stops.shp` at 20 km/h.
- `Y`: feeder-bus travel time from each bus stop to the best LRT transfer stop, estimated along `N07-11_09_GML.shp` bus route lines at 20 km/h when available.
- `X`: remaining walking time, calculated as threshold minus `Y + Z` and expanded along the road network when an N13 road layer is available.

The road-network step avoids simple circular buffers when road lines are present. If required local data are missing, the notebook reports the missing inputs rather than silently fabricating results.


## Expected local/attached data

Place complete shapefile sets under `data/`, including `.shp`, `.shx`, `.dbf`, and `.prj` when available. The bus route map attached to the request should be available as:

- `data/N07-11_09_GML.shp` (plus sidecars)

The workflow also searches common alternatives recursively under `data/` for:

- `lrt_stops.shp` (required)
- `P11-22_09.shp` bus stop points (recommended)
- `N13-24_5439.shp` road lines (recommended for road-network walking isochrones)


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))

import kiyohara_transit_isochrone as iso


In [ ]:
# Configure the destination stop.
# Use a more specific expression if the target should be a particular LRT stop,
# for example: r"清原地区市民センター前" or r"芳賀・高根沢工業団地".
DATA_DIR = "data"
OUTPUT_DIR = "outputs/kiyohara_isochrone"
TARGET_STOP_REGEX = r"清原|工業団地"
THRESHOLDS = [30, 60]


In [ ]:
# Input discovery and sidecar check.
# This should print the resolved local layers before running the analysis.
iso.check_inputs(Path(DATA_DIR))


In [ ]:
# Run the full 30/60-minute isochrone analysis.
args = iso.parse_args([
    "--data-dir", DATA_DIR,
    "--output-dir", OUTPUT_DIR,
    "--target-stop-regex", TARGET_STOP_REGEX,
    "--thresholds", *map(str, THRESHOLDS),
])
iso.run_analysis(args)


## Outputs

Expected outputs are written under `outputs/kiyohara_isochrone/`:

- `kiyohara_transit_isochrones.gpkg` with `isochrone_30min`, `isochrone_60min`, `access_nodes`, and `lrt_travel_times` layers
- `shapefiles/kiyohara_isochrone_30min.shp` and `shapefiles/kiyohara_isochrone_60min.shp`
- `tables/kiyohara_transit_access_nodes.csv` and `tables/kiyohara_lrt_travel_times.csv`
- `figures/kiyohara_transit_isochrones.png`
